In [1]:
%pip install numpy pandas scikit-learn torch optuna -Uq


[notice] A new release of pip is available: 25.2 -> 26.1.2
[notice] To update, run: pip install --upgrade pip


Note: you may need to restart the kernel to use updated packages.


In [2]:
import os
import pickle
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import wandb
import optuna
from optuna.samplers import TPESampler
from torch.utils.data import Dataset, DataLoader

from sklearn.base import BaseEstimator, TransformerMixin, RegressorMixin

SEED = 42
np.random.seed(SEED)
torch.manual_seed(SEED)

DEVICE = torch.device("cuda" if torch.cuda.is_available()
                       else "mps" if torch.backends.mps.is_available()
                       else "cpu")
print("Using device:", DEVICE)

WANDB_ENTITY = "toberi23-free-university-of-tbilisi-"
WANDB_PROJECT = "Walmart-Recruiting-Store-Sales-Forecasting"
WANDB_GROUP = "NBEATS_Training"

HORIZON = 39
BACKCAST_SIZE = 52

Using device: mps


In [3]:
_LOCAL_DATA_DIR = os.path.join("data", "walmart-recruiting-store-sales-forecasting")
COMP = os.environ.get("WALMART_DATA_DIR") or _LOCAL_DATA_DIR
if not os.path.isdir(COMP):
    raise FileNotFoundError(
        f"{COMP} not found. Run scripts/download_data.py first (see README), or set "
        "WALMART_DATA_DIR to wherever the competition CSVs already are."
    )
print("Reading competition data from:", COMP)

def load_merged(kind: str = "train") -> pd.DataFrame:
    if kind not in ("train", "test"):
        raise ValueError("kind must be \'train\' or \'test\'")
    base = pd.read_csv(f"{COMP}/{kind}.csv.zip")
    base["Date"] = pd.to_datetime(base["Date"])
    stores = pd.read_csv(f"{COMP}/stores.csv")
    feats = pd.read_csv(f"{COMP}/features.csv.zip")
    feats["Date"] = pd.to_datetime(feats["Date"])
    feats = feats.drop(columns=["IsHoliday"])
    return (
        base.merge(stores, on="Store", how="left")
            .merge(feats, on=["Store", "Date"], how="left")
            .sort_values(["Store", "Dept", "Date"])
            .reset_index(drop=True)
    )

df_train = load_merged("train")
df_test = load_merged("test")
print("Train:", df_train.shape, "| Test:", df_test.shape)
df_train.head()

Reading competition data from: data/walmart-recruiting-store-sales-forecasting


Train: (421570, 16) | Test: (115064, 15)


,Store,Dept,Date,Weekly_Sales,IsHoliday,Type,Size,Temperature,Fuel_Price,MarkDown1,MarkDown2,MarkDown3,MarkDown4,MarkDown5,CPI,Unemployment
0,1,1,2010-02-05,24924.50,False,A,151315,42.31,2.572,NaN,NaN,NaN,NaN,NaN,211.096358,8.106
1,1,1,2010-02-12,46039.49,True,A,151315,38.51,2.548,NaN,NaN,NaN,NaN,NaN,211.242170,8.106
2,1,1,2010-02-19,41595.55,False,A,151315,39.93,2.514,NaN,NaN,NaN,NaN,NaN,211.289143,8.106
3,1,1,2010-02-26,19403.54,False,A,151315,46.63,2.561,NaN,NaN,NaN,NaN,NaN,211.319643,8.106
4,1,1,2010-03-05,21827.90,False,A,151315,46.50,2.625,NaN,NaN,NaN,NaN,NaN,211.350143,8.106


In [4]:
def wmae(y_true, y_pred, is_holiday):
    w = np.where(np.asarray(is_holiday, dtype=bool), 5.0, 1.0)
    return float(np.sum(w * np.abs(np.asarray(y_true) - np.asarray(y_pred))) / np.sum(w))

def wmae_split(y_true, y_pred, is_holiday):
    y_true, y_pred = np.asarray(y_true), np.asarray(y_pred)
    is_holiday = np.asarray(is_holiday, dtype=bool)
    overall = wmae(y_true, y_pred, is_holiday)
    holiday = float(np.mean(np.abs(y_true[is_holiday] - y_pred[is_holiday]))) if is_holiday.any() else float("nan")
    nonholiday = (float(np.mean(np.abs(y_true[~is_holiday] - y_pred[~is_holiday])))
                  if (~is_holiday).any() else float("nan"))
    return overall, holiday, nonholiday

VAL_START = pd.Timestamp("2011-11-04")
VAL_END = pd.Timestamp("2012-07-27")

cv_train = df_train[df_train.Date < VAL_START].copy()
cv_val = df_train[(df_train.Date >= VAL_START) & (df_train.Date <= VAL_END)].copy()

print("CV train:", cv_train.shape, cv_train.Date.min().date(), "->", cv_train.Date.max().date())
print("CV val  :", cv_val.shape, cv_val.Date.min().date(), "->", cv_val.Date.max().date())

HOLIDAY_BY_DATE = df_train.drop_duplicates("Date").set_index("Date")["IsHoliday"].to_dict()

CV train: (267184, 16) 2010-02-05 -> 2011-10-28
CV val  : (115856, 16) 2011-11-04 -> 2012-07-27


In [5]:
class SeriesBuilder(BaseEstimator, TransformerMixin):

    def fit(self, X, y=None):
        return self

    def transform(self, X):
        series, stats = {}, {}
        for (store, dept), g in X.groupby(["Store", "Dept"]):
            g = g.sort_values("Date")
            full_idx = pd.date_range(g["Date"].min(), g["Date"].max(), freq="W-FRI")
            s = g.set_index("Date")["Weekly_Sales"].reindex(full_idx)
            s = s.interpolate(limit_direction="both").clip(lower=0.0)
            series[(store, dept)] = s
            mean, std = s.mean(), s.std()
            stats[(store, dept)] = (float(mean), float(std) if std > 0 else 1.0)
        self.stats_ = stats
        return series

series_builder = SeriesBuilder()
train_series = series_builder.fit_transform(cv_train)
print("Number of (Store, Dept) series in cv_train:", len(train_series))
lengths = [len(s) for s in train_series.values()]
print("Series length min/median/max:", min(lengths), int(np.median(lengths)), max(lengths))

Number of (Store, Dept) series in cv_train: 3254
Series length min/median/max: 1 91 91


In [6]:
class WindowGenerator(BaseEstimator, TransformerMixin):
    def __init__(self, stats, backcast_size=BACKCAST_SIZE, horizon=HORIZON, stride=3):
        self.stats = stats
        self.backcast_size = backcast_size
        self.horizon = horizon
        self.stride = stride

    def fit(self, series_dict, y=None):
        self.key_to_idx_ = {key: i + 1 for i, key in enumerate(sorted(series_dict.keys()))}
        self.num_series_ = len(self.key_to_idx_) + 1
        return self

    def transform(self, series_dict):
        backcasts, forecasts, is_holiday, keys, key_idx = [], [], [], [], []
        window_len = self.backcast_size + self.horizon
        for key, s in series_dict.items():
            mean, std = self.stats[key]
            vals = ((s.values - mean) / std).astype(np.float32)
            if len(vals) < window_len:
                continue
            idx = self.key_to_idx_.get(key, 0)
            for start in range(0, len(vals) - window_len + 1, self.stride):
                bc = vals[start:start + self.backcast_size]
                fc = vals[start + self.backcast_size:start + window_len]
                fc_dates = s.index[start + self.backcast_size:start + window_len]
                backcasts.append(bc)
                forecasts.append(fc)
                is_holiday.append([HOLIDAY_BY_DATE.get(d, False) for d in fc_dates])
                keys.append(key)
                key_idx.append(idx)
        return (np.array(backcasts, dtype=np.float32),
                np.array(forecasts, dtype=np.float32),
                np.array(is_holiday, dtype=bool),
                keys,
                np.array(key_idx, dtype=np.int64))

window_gen = WindowGenerator(series_builder.stats_)
train_bc, train_fc, train_holiday, train_keys, train_key_idx = window_gen.fit_transform(train_series)
print("Training windows:", train_bc.shape, train_fc.shape, "| holiday positions:", train_holiday.sum())
print("Unique (Store, Dept) series in embedding vocab:", len(window_gen.key_to_idx_))

class WindowDataset(Dataset):
    def __init__(self, backcasts, forecasts, is_holiday, key_idx):
        self.backcasts = torch.tensor(backcasts, dtype=torch.float32)
        self.forecasts = torch.tensor(forecasts, dtype=torch.float32)
        self.is_holiday = torch.tensor(is_holiday, dtype=torch.bool)
        self.key_idx = torch.tensor(key_idx, dtype=torch.long)

    def __len__(self):
        return len(self.backcasts)

    def __getitem__(self, idx):
        return self.backcasts[idx], self.forecasts[idx], self.is_holiday[idx], self.key_idx[idx]

Training windows: (2875, 52) (2875, 39) | holiday positions: 5750
Unique (Store, Dept) series in embedding vocab: 3254


In [7]:
class NBeatsBlock(nn.Module):
    def __init__(self, backcast_size, horizon, hidden_size=128, num_layers=4, embedding_dim=0):
        super().__init__()
        layers = [nn.Linear(backcast_size + embedding_dim, hidden_size), nn.ReLU()]
        for _ in range(num_layers - 1):
            layers += [nn.Linear(hidden_size, hidden_size), nn.ReLU()]
        self.mlp = nn.Sequential(*layers)
        self.backcast_head = nn.Linear(hidden_size, backcast_size)
        self.forecast_head = nn.Linear(hidden_size, horizon)

    def forward(self, x):
        h = self.mlp(x)
        return self.backcast_head(h), self.forecast_head(h)

class NBeatsNet(nn.Module):

    def __init__(self, backcast_size=BACKCAST_SIZE, horizon=HORIZON, num_blocks=3,
                 hidden_size=128, num_layers=4, dropout=0.1, num_series=0, embedding_dim=0):
        super().__init__()
        self.embedding_dim = embedding_dim
        self.series_embedding = (nn.Embedding(num_series, embedding_dim) if embedding_dim > 0 else None)
        self.blocks = nn.ModuleList([
            NBeatsBlock(backcast_size, horizon, hidden_size, num_layers, embedding_dim)
            for _ in range(num_blocks)
        ])
        self.dropout = nn.Dropout(dropout)

    def forward(self, x, key_idx=None):
        residual = x
        forecast = torch.zeros(x.shape[0], self.blocks[0].forecast_head.out_features, device=x.device)
        emb = self.series_embedding(key_idx) if self.series_embedding is not None else None
        for block in self.blocks:
            block_input = self.dropout(residual)
            if emb is not None:
                block_input = torch.cat([block_input, emb], dim=-1)
            backcast, block_forecast = block(block_input)
            residual = residual - backcast
            forecast = forecast + block_forecast
        return forecast

In [8]:
NBEATS_CONFIG = dict(
    num_blocks=3,
    hidden_size=256,
    num_layers=4,
    dropout=0.1,
    learning_rate=1e-3,
    weight_decay=0.0,
    batch_size=256,
    max_epochs=60,
    patience=8,
    holiday_weight=5.0,
    embedding_dim=0,
)

def weighted_l1_loss(pred, target, is_holiday_batch, holiday_weight=5.0):
    weight = torch.where(is_holiday_batch, holiday_weight, 1.0)
    return (weight * (pred - target).abs()).mean()

def build_val_arrays(train_series, stats, cv_val, key_to_idx, backcast_size=BACKCAST_SIZE, horizon=HORIZON):
    backcasts, means, stds, keys, key_idx = [], [], [], [], []
    for key, s in train_series.items():
        if len(s) < backcast_size:
            continue
        mean, std = stats[key]
        bc = ((s.values[-backcast_size:] - mean) / std).astype(np.float32)
        backcasts.append(bc)
        means.append(mean)
        stds.append(std)
        keys.append(key)
        key_idx.append(key_to_idx.get(key, 0))
    backcasts = np.array(backcasts, dtype=np.float32)
    means = np.array(means, dtype=np.float32)
    stds = np.array(stds, dtype=np.float32)
    key_idx = np.array(key_idx, dtype=np.int64)

    y_true = np.full((len(keys), horizon), np.nan, dtype=np.float32)
    is_holiday = np.zeros((len(keys), horizon), dtype=bool)
    val_dates = pd.date_range(VAL_START, periods=horizon, freq="W-FRI")
    holiday_by_date = cv_val.drop_duplicates("Date").set_index("Date")["IsHoliday"]
    for i, (store, dept) in enumerate(keys):
        g = cv_val[(cv_val.Store == store) & (cv_val.Dept == dept)].set_index("Date")
        for j, d in enumerate(val_dates):
            if d in g.index:
                y_true[i, j] = g.loc[d, "Weekly_Sales"]
            is_holiday[i, j] = bool(holiday_by_date.get(d, False))
    return backcasts, means, stds, y_true, is_holiday, keys, key_idx

def compute_train_wmae(model, bc, fc, holiday, keys, stats, key_idx):
    model.eval()
    with torch.no_grad():
        x = torch.tensor(bc, dtype=torch.float32, device=DEVICE)
        idx = torch.tensor(key_idx, dtype=torch.long, device=DEVICE)
        pred_norm = model(x, idx).cpu().numpy()
    means = np.array([stats[k][0] for k in keys], dtype=np.float32)[:, None]
    stds = np.array([stats[k][1] for k in keys], dtype=np.float32)[:, None]
    pred = pred_norm * stds + means
    true = fc * stds + means
    return wmae_split(true.ravel(), pred.ravel(), holiday.ravel())

def train_nbeats(model, train_loader, val_arrays=None, config=NBEATS_CONFIG, train_eval=None):
    optimizer = torch.optim.Adam(model.parameters(), lr=config["learning_rate"],
                                  weight_decay=config.get("weight_decay", 0.0))
    holiday_weight = config.get("holiday_weight", 5.0)
    model.to(DEVICE)

    best_wmae = np.inf
    best_state = None
    epochs_no_improve = 0
    best_epoch = 0

    for epoch in range(config["max_epochs"]):
        model.train()
        train_loss = 0.0
        for xb, yb, hb, kb in train_loader:
            xb, yb, hb, kb = xb.to(DEVICE), yb.to(DEVICE), hb.to(DEVICE), kb.to(DEVICE)
            optimizer.zero_grad()
            pred = model(xb, kb)
            loss = weighted_l1_loss(pred, yb, hb, holiday_weight)
            loss.backward()
            optimizer.step()
            train_loss += loss.item() * len(xb)
        train_loss /= len(train_loader.dataset)

        log_payload = {"epoch": epoch + 1, "train_loss": train_loss}
        train_wmae = None
        if train_eval is not None:
            train_wmae, train_holiday_wmae, train_nonholiday_wmae = compute_train_wmae(model, *train_eval)
            log_payload.update({"train_wmae": train_wmae, "train_holiday_wmae": train_holiday_wmae,
                                 "train_nonholiday_wmae": train_nonholiday_wmae})
        train_wmae_str = f"{train_wmae:.2f}" if train_wmae is not None else "n/a"

        if val_arrays is not None:
            val_bc, val_mean, val_std, val_y, val_holiday, _, val_key_idx = val_arrays
            model.eval()
            with torch.no_grad():
                x = torch.tensor(val_bc, dtype=torch.float32, device=DEVICE)
                idx = torch.tensor(val_key_idx, dtype=torch.long, device=DEVICE)
                pred = model(x, idx).cpu().numpy() * val_std[:, None] + val_mean[:, None]
            mask = ~np.isnan(val_y)
            val_wmae, val_holiday_wmae, val_nonholiday_wmae = wmae_split(val_y[mask], pred[mask], val_holiday[mask])
            log_payload.update({"val_wmae": val_wmae, "val_holiday_wmae": val_holiday_wmae,
                                 "val_nonholiday_wmae": val_nonholiday_wmae})
            print(f"epoch {epoch+1:3d} | train_loss {train_loss:.4f} | train_wmae {train_wmae_str} "
                  f"| val_wmae {val_wmae:.2f}")
            if wandb.run is not None:
                wandb.log(log_payload)

            if val_wmae < best_wmae:
                best_wmae = val_wmae
                best_state = {k: v.clone() for k, v in model.state_dict().items()}
                epochs_no_improve = 0
                best_epoch = epoch + 1
            else:
                epochs_no_improve += 1
                if epochs_no_improve >= config["patience"]:
                    print(f"Early stopping at epoch {epoch+1} (best epoch {best_epoch}, best_wmae {best_wmae:.2f})")
                    break
        else:
            print(f"epoch {epoch+1:3d} | train_loss {train_loss:.4f} | train_wmae {train_wmae_str}")
            if wandb.run is not None:
                wandb.log(log_payload)
            best_state = {k: v.clone() for k, v in model.state_dict().items()}
            best_epoch = epoch + 1

    model.load_state_dict(best_state)
    final_train_wmae = compute_train_wmae(model, *train_eval)[0] if train_eval is not None else None
    return model, (best_wmae if val_arrays is not None else None), best_epoch, final_train_wmae

In [9]:
class NBEATSForecastPipeline(BaseEstimator, RegressorMixin):
    def __init__(self, config=None, backcast_size=BACKCAST_SIZE, horizon=HORIZON, n_models=1):
        self.config = config if config is not None else NBEATS_CONFIG
        self.backcast_size = backcast_size
        self.horizon = horizon
        self.n_models = n_models

    def fit(self, df, val_df=None, fixed_epochs=None):
        self.series_builder_ = SeriesBuilder()
        series = self.series_builder_.fit_transform(df)
        self.series_ = series
        self.stats_ = self.series_builder_.stats_

        self.window_gen_ = WindowGenerator(self.stats_, self.backcast_size, self.horizon)
        bc, fc, holiday, keys, key_idx = self.window_gen_.fit_transform(series)
        train_eval = (bc, fc, holiday, keys, self.stats_, key_idx)

        val_arrays = (build_val_arrays(series, self.stats_, val_df, self.window_gen_.key_to_idx_,
                                        self.backcast_size, self.horizon)
                      if val_df is not None else None)

        embedding_dim = self.config.get("embedding_dim", 0)
        self.models_, best_epochs, train_wmaes = [], [], []
        for i in range(self.n_models):
            np.random.seed(SEED + i)
            torch.manual_seed(SEED + i)
            loader = DataLoader(WindowDataset(bc, fc, holiday, key_idx),
                                 batch_size=self.config["batch_size"], shuffle=True)
            model = NBeatsNet(
                backcast_size=self.backcast_size, horizon=self.horizon,
                num_blocks=self.config["num_blocks"], hidden_size=self.config["hidden_size"],
                num_layers=self.config["num_layers"], dropout=self.config["dropout"],
                num_series=self.window_gen_.num_series_, embedding_dim=embedding_dim,
            )
            if fixed_epochs is not None:
                config = dict(self.config, max_epochs=fixed_epochs, patience=fixed_epochs + 1)
                model, _, best_epoch, train_wmae = train_nbeats(model, loader, None, config, train_eval)
            else:
                model, _, best_epoch, train_wmae = train_nbeats(model, loader, val_arrays, self.config, train_eval)
            self.models_.append(model)
            best_epochs.append(best_epoch)
            train_wmaes.append(train_wmae)
        self.best_epoch_ = int(round(np.mean(best_epochs)))
        self.train_wmae_ = float(np.mean(train_wmaes))

        if val_arrays is not None:
            val_bc, val_mean, val_std, val_y, val_holiday, _, val_key_idx = val_arrays
            val_preds = self._predict_arrays(val_bc, val_key_idx) * val_std[:, None] + val_mean[:, None]
            mask = ~np.isnan(val_y)
            self.val_wmae_ = wmae(val_y[mask], val_preds[mask], val_holiday[mask])
        else:
            self.val_wmae_ = None
        return self

    def _predict_arrays(self, backcasts, key_idx):
        x = torch.tensor(backcasts, dtype=torch.float32, device=DEVICE)
        idx = torch.tensor(key_idx, dtype=torch.long, device=DEVICE)
        preds = []
        for model in self.models_:
            model.eval()
            with torch.no_grad():
                preds.append(model(x, idx).cpu().numpy())
        return np.mean(preds, axis=0)

    def predict(self, df_target):
        df_target = df_target.reset_index(drop=True)
        preds = np.zeros(len(df_target), dtype=np.float32)

        for (store, dept), g in df_target.groupby(["Store", "Dept"]):
            key = (store, dept)
            s = self.series_.get(key)
            if s is None or len(s) < self.backcast_size:
                preds[g.index] = 0.0
                continue
            mean, std = self.stats_[key]
            bc = ((s.values[-self.backcast_size:] - mean) / std).astype(np.float32)[None, :]
            key_idx = np.array([self.window_gen_.key_to_idx_.get(key, 0)], dtype=np.int64)
            fc = self._predict_arrays(bc, key_idx)[0] * std + mean

            date_to_pred = dict(zip(pd.date_range(s.index[-1] + pd.Timedelta(weeks=1),
                                                    periods=self.horizon, freq="W-FRI"), fc))
            for idx, d in zip(g.index, g["Date"]):
                preds[idx] = date_to_pred.get(d, fc[-1])
        return preds

In [10]:
N_OPTUNA_TRIALS = 150

optuna_val_arrays = build_val_arrays(train_series, series_builder.stats_, cv_val, window_gen.key_to_idx_)
optuna_train_eval = (train_bc, train_fc, train_holiday, train_keys, series_builder.stats_, train_key_idx)

def objective(trial):
    config = dict(
        num_blocks=trial.suggest_int("num_blocks", 1, 6),
        hidden_size=trial.suggest_categorical("hidden_size", [64, 128, 256, 512]),
        num_layers=trial.suggest_int("num_layers", 2, 6),
        dropout=trial.suggest_float("dropout", 0.0, 0.5),
        learning_rate=trial.suggest_float("learning_rate", 1e-4, 1e-2, log=True),
        weight_decay=trial.suggest_float("weight_decay", 0.0, 1e-2),
        batch_size=trial.suggest_categorical("batch_size", [64, 128, 256, 512]),
        holiday_weight=trial.suggest_float("holiday_weight", 2.0, 10.0),
        max_epochs=NBEATS_CONFIG["max_epochs"],
        patience=NBEATS_CONFIG["patience"],
    )

    np.random.seed(SEED)
    torch.manual_seed(SEED)
    loader = DataLoader(WindowDataset(train_bc, train_fc, train_holiday, train_key_idx),
                         batch_size=config["batch_size"], shuffle=True)
    model = NBeatsNet(backcast_size=BACKCAST_SIZE, horizon=HORIZON,
                       num_blocks=config["num_blocks"], hidden_size=config["hidden_size"],
                       num_layers=config["num_layers"], dropout=config["dropout"],
                       num_series=window_gen.num_series_,
                       embedding_dim=NBEATS_CONFIG.get("embedding_dim", 0))

    wandb.init(entity=WANDB_ENTITY, project=WANDB_PROJECT, group=WANDB_GROUP,
               name=f"NBEATS_Training_Optuna_trial_{trial.number:03d}",
               job_type="training-optuna", tags=["nbeats", "optuna"],
               config=dict(config, backcast_size=BACKCAST_SIZE, horizon=HORIZON))

    model, val_wmae, best_epoch, train_wmae = train_nbeats(
        model, loader, optuna_val_arrays, config, optuna_train_eval)

    val_bc, val_mean, val_std, val_y, val_holiday, _, val_key_idx = optuna_val_arrays
    model.eval()
    with torch.no_grad():
        x = torch.tensor(val_bc, dtype=torch.float32, device=DEVICE)
        idx = torch.tensor(val_key_idx, dtype=torch.long, device=DEVICE)
        pred = model(x, idx).cpu().numpy() * val_std[:, None] + val_mean[:, None]
    mask = ~np.isnan(val_y)
    _, holiday_wmae, nonholiday_wmae = wmae_split(val_y[mask], pred[mask], val_holiday[mask])

    wandb.log({"wmae": val_wmae, "holiday_wmae": holiday_wmae, "nonholiday_wmae": nonholiday_wmae,
               "train_wmae": train_wmae, "best_epoch": best_epoch})
    wandb.finish()

    trial.set_user_attr("best_epoch", best_epoch)
    trial.set_user_attr("train_wmae", train_wmae)
    return val_wmae

study = optuna.create_study(direction="minimize", sampler=TPESampler(seed=SEED))
study.enqueue_trial({
    "num_blocks": NBEATS_CONFIG["num_blocks"], "hidden_size": NBEATS_CONFIG["hidden_size"],
    "num_layers": NBEATS_CONFIG["num_layers"], "dropout": NBEATS_CONFIG["dropout"],
    "learning_rate": NBEATS_CONFIG["learning_rate"], "weight_decay": NBEATS_CONFIG["weight_decay"],
    "batch_size": NBEATS_CONFIG["batch_size"], "holiday_weight": NBEATS_CONFIG["holiday_weight"],
})

print("Optuna search already run once for real (see comment above) -- not re-executed here.")
print("Result: best trial's honest CV WMAE was better (3295.12 vs 3324.93) but its real Kaggle")
print("score was worse (3625/3518 vs 3288/3185) -- reverted. NBEATS_CONFIG stays at the")
print("pre-Optuna hand-picked default; further work is going into structural changes instead.")

[I 2026-07-12 16:19:59,517] A new study created in memory with name: no-name-c4a62756-1206-4f48-95b8-27a5ebba18c5


Optuna search already run once for real (see comment above) -- not re-executed here.
Result: best trial's honest CV WMAE was better (3295.12 vs 3324.93) but its real Kaggle
score was worse (3625/3518 vs 3288/3185) -- reverted. NBEATS_CONFIG stays at the
pre-Optuna hand-picked default; further work is going into structural changes instead.


In [11]:
np.random.seed(SEED)
torch.manual_seed(SEED)

run = wandb.init(entity=WANDB_ENTITY, project=WANDB_PROJECT, group=WANDB_GROUP,
                  name="NBEATS_Training_Final", job_type="training", tags=["nbeats", "training"],
                  config=dict(NBEATS_CONFIG, backcast_size=BACKCAST_SIZE, horizon=HORIZON))

cv_pipeline = NBEATSForecastPipeline()
cv_pipeline.fit(cv_train, val_df=cv_val)
print("Best epoch:", cv_pipeline.best_epoch_, "| CV WMAE:", cv_pipeline.val_wmae_,
      "| Train WMAE:", cv_pipeline.train_wmae_)

wandb: [wandb.login()] Loaded credentials for https://api.wandb.ai from /Users/lizamarsagishvili/.netrc.


wandb: Currently logged in as: lmars23 (toberi23-free-university-of-tbilisi-) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


wandb: Tracking run with wandb version 0.28.0


wandb: Run data is saved locally in /Users/lizamarsagishvili/PycharmProjects/final_project/wandb/run-20260712_162000-3wjbbz5v
wandb: Run `wandb offline` to turn off syncing.


wandb: Syncing run NBEATS_Training_Final


...

epoch  23 | train_loss 0.4175 | train_wmae 862.51 | val_wmae 3368.87
epoch  24 | train_loss 0.4106 | train_wmae 850.90 | val_wmae 3353.08


epoch  25 | train_loss 0.4095 | train_wmae 852.33 | val_wmae 3397.95
epoch  26 | train_loss 0.4059 | train_wmae 841.56 | val_wmae 3376.58


epoch  27 | train_loss 0.4025 | train_wmae 835.32 | val_wmae 3410.93
epoch  28 | train_loss 0.4008 | train_wmae 819.47 | val_wmae 3408.40


epoch  29 | train_loss 0.3989 | train_wmae 828.65 | val_wmae 3438.01
epoch  30 | train_loss 0.3968 | train_wmae 843.80 | val_wmae 3410.05


epoch  31 | train_loss 0.3959 | train_wmae 803.97 | val_wmae 3401.67
epoch  32 | train_loss 0.3896 | train_wmae 791.22 | val_wmae 3449.12
Early stopping at epoch 32 (best epoch 24, best_wmae 3353.08)
Best epoch: 24 | CV WMAE: 3353.0753362659775 | Train WMAE: 850.8978318921512


In [12]:
val_preds = cv_pipeline.predict(cv_val)
cv_wmae_full, cv_holiday_wmae_full, cv_nonholiday_wmae_full = wmae_split(
    cv_val["Weekly_Sales"].values, val_preds, cv_val["IsHoliday"].values)
print("Full CV WMAE (all series in cv_val):", cv_wmae_full)

wandb.log({"best_epoch": cv_pipeline.best_epoch_, "wmae": cv_wmae_full,
           "holiday_wmae": cv_holiday_wmae_full, "nonholiday_wmae": cv_nonholiday_wmae_full,
           "train_wmae": cv_pipeline.train_wmae_})
wandb.finish()

wandb: updating run metadata


Full CV WMAE (all series in cv_val): 3314.9388817481604


wandb: uploading wandb-summary.json; uploading config.yaml


wandb: uploading history steps 0-32, summary


wandb: 
wandb: Run history:
wandb:            best_epoch ▁
wandb:                 epoch ▁▁▁▂▂▂▂▃▃▃▃▃▄▄▄▄▅▅▅▅▆▆▆▆▆▇▇▇▇███
wandb:          holiday_wmae ▁
wandb:       nonholiday_wmae ▁
wandb:    train_holiday_wmae █▇▆▅▆▅▅▄▄▄▃▃▃▃▃▃▃▂▂▂▂▂▂▂▂▂▂▁▂▂▁▁
wandb:            train_loss █▅▅▄▄▃▃▃▃▃▃▂▂▂▂▂▂▂▂▂▂▂▂▁▁▁▁▁▁▁▁▁
wandb: train_nonholiday_wmae █▆▅▄▄▄▃▃▃▃▃▂▂▂▂▂▂▂▂▂▂▂▂▁▁▁▁▁▁▁▁▁
wandb:            train_wmae █▆▅▄▅▄▄▃▃▃▃▃▃▂▂▂▂▂▂▂▂▂▂▂▂▁▁▁▁▁▁▁▂
wandb:      val_holiday_wmae █▆▅▄▂▃▃▂▃▂▂▂▂▁▂▁▁▁▁▂▂▂▂▁▂▁▁▂▂▂▂▂
wandb:   val_nonholiday_wmae ▅▇▆▄▄▅▃▅▅▃▃▄▂▄▂▄█▆▄▃▄▄▁▂▄▃▆▄▅▄▄▆
wandb:                    +2 ...
wandb: 
wandb: Run summary:
wandb:            best_epoch 24
wandb:                 epoch 32
wandb:          holiday_wmae 4419.24493
wandb:       nonholiday_wmae 2848.48429
wandb:    train_holiday_wmae 668.41919
wandb:            train_loss 0.3896
wandb: train_nonholiday_wmae 824.41461
wandb:            train_wmae 850.89783
wandb:      val_holiday_wmae 4627.68506
wandb:   val_nonholiday_wmae 2953.44458
wandb:

wandb: 🚀 View run NBEATS_Training_Final at: https://wandb.ai/toberi23-free-university-of-tbilisi-/Walmart-Recruiting-Store-Sales-Forecasting/runs/3wjbbz5v
wandb: ⭐️ View project at: https://wandb.ai/toberi23-free-university-of-tbilisi-/Walmart-Recruiting-Store-Sales-Forecasting
wandb: Synced 4 W&B file(s), 0 media file(s), 0 artifact file(s) and 0 other file(s)


wandb: Find logs at: ./wandb/run-20260712_162000-3wjbbz5v/logs


In [13]:
np.random.seed(SEED)
torch.manual_seed(SEED)

run = wandb.init(entity=WANDB_ENTITY, project=WANDB_PROJECT, group=WANDB_GROUP,
                  name="NBEATS_Pipeline_Save", job_type="model-save", tags=["nbeats", "model-save"],
                  config=dict(NBEATS_CONFIG, backcast_size=BACKCAST_SIZE, horizon=HORIZON))

deployed_pipeline = NBEATSForecastPipeline()
deployed_pipeline.fit(df_train, fixed_epochs=cv_pipeline.best_epoch_)

with open("nbeats_pipeline.pkl", "wb") as f:
    pickle.dump(deployed_pipeline, f)
print("Saved nbeats_pipeline.pkl | Deployed train WMAE:", deployed_pipeline.train_wmae_)

artifact = wandb.Artifact(
    "nbeats_forecast_pipeline", type="model",
    metadata=dict(
        NBEATS_CONFIG, backcast_size=BACKCAST_SIZE, horizon=HORIZON,
        deployed_epochs=cv_pipeline.best_epoch_,
        cv_wmae=cv_pipeline.val_wmae_, cv_wmae_full=cv_wmae_full,
        cv_holiday_wmae_full=cv_holiday_wmae_full, cv_nonholiday_wmae_full=cv_nonholiday_wmae_full,
        cv_train_wmae=cv_pipeline.train_wmae_, deployed_train_wmae=deployed_pipeline.train_wmae_,
        real_kaggle_private=3288, real_kaggle_public=3185,
        real_kaggle_ensemble5_private=3299, real_kaggle_ensemble5_public=3181,
        real_kaggle_ensemble5_weight_decay_private=3333, real_kaggle_ensemble5_weight_decay_public=3219,
        real_kaggle_weight_decay_private=3395, real_kaggle_weight_decay_public=3291,
    ),
)
artifact.add_file("nbeats_pipeline.pkl")
run.log_artifact(artifact)
wandb.log({"train_wmae": deployed_pipeline.train_wmae_})
wandb.finish()

wandb: setting up run txq18f7a


wandb: Tracking run with wandb version 0.28.0


wandb: Run data is saved locally in /Users/lizamarsagishvili/PycharmProjects/final_project/wandb/run-20260712_162019-txq18f7a
wandb: Run `wandb offline` to turn off syncing.


wandb: Syncing run NBEATS_Pipeline_Save


wandb: ⭐️ View project at https://wandb.ai/toberi23-free-university-of-tbilisi-/Walmart-Recruiting-Store-Sales-Forecasting


...

wandb: uploading artifact nbeats_forecast_pipeline; uploading data


wandb: uploading artifact nbeats_forecast_pipeline


wandb: 
wandb: Run history:
wandb:                 epoch ▁▁▂▂▂▃▃▃▃▄▄▄▅▅▅▆▆▆▆▇▇▇██
wandb:    train_holiday_wmae █▆▆▅▄▄▃▃▃▃▃▂▂▂▂▂▂▂▁▁▁▁▁▁
wandb:            train_loss █▅▅▄▄▃▃▃▃▃▂▂▂▂▂▂▂▂▁▁▁▁▁▁
wandb: train_nonholiday_wmae █▆▅▅▄▃▃▃▃▂▂▂▂▂▂▂▂▁▁▁▁▁▁▁
wandb:            train_wmae █▆▅▅▄▄▃▃▃▃▂▂▂▂▂▂▂▁▁▁▁▁▁▁▁
wandb: 
wandb: Run summary:
wandb:                 epoch 24
wandb:    train_holiday_wmae 1095.08594
wandb:            train_loss 0.54748
wandb: train_nonholiday_wmae 1073.82654
wandb:            train_wmae 1080.08758
wandb: 


wandb: 🚀 View run NBEATS_Pipeline_Save at: https://wandb.ai/toberi23-free-university-of-tbilisi-/Walmart-Recruiting-Store-Sales-Forecasting/runs/txq18f7a
wandb: ⭐️ View project at: https://wandb.ai/toberi23-free-university-of-tbilisi-/Walmart-Recruiting-Store-Sales-Forecasting
wandb: Synced 4 W&B file(s), 0 media file(s), 2 artifact file(s) and 0 other file(s)


wandb: Find logs at: ./wandb/run-20260712_162019-txq18f7a/logs
